# 재무 공식 검증 및 파생 변수 생성

## 목표
1. 재무제표 항목 간 관계 검증
2. 재무비율 계산 검증
3. 파생 변수(Feature) 생성
4. 데이터 품질 개선

## 참고 문서
- `docs/column_dictionary.md`: 컬럼 설명 및 공식

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [2]:
# 데이터 로드
df = pd.read_csv('../data/기업신용평가정보_합성데이터.csv', encoding='cp949')
print(f"데이터 shape: {df.shape}")

데이터 shape: (600000, 159)


## 1. 재무상태표 공식 검증

### 1.1 총자산 = 유동자산 + 비유동자산

In [3]:
# 공식 검증
if all(col in df.columns for col in ['총자산', '유동자산', '비유동자산']):
    df['총자산_계산1'] = df['유동자산'] + df['비유동자산']
    df['총자산_오차1'] = df['총자산'] - df['총자산_계산1']
    df['총자산_오차율1'] = (df['총자산_오차1'].abs() / df['총자산'] * 100)
    
    print("=== 총자산 = 유동자산 + 비유동자산 ===")
    print(f"평균 오차: {df['총자산_오차1'].mean():,.0f}")
    print(f"평균 오차율: {df['총자산_오차율1'].mean():.4f}%")
    print(f"\n오차율 1% 초과: {(df['총자산_오차율1'] > 1).sum():,}건 ({(df['총자산_오차율1'] > 1).sum() / len(df) * 100:.2f}%)")
    print(f"오차율 5% 초과: {(df['총자산_오차율1'] > 5).sum():,}건 ({(df['총자산_오차율1'] > 5).sum() / len(df) * 100:.2f}%)")
    
    # 오차 분포 시각화
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    df['총자산_오차율1'].hist(bins=100, range=(0, 10), edgecolor='black')
    plt.axvline(x=1, color='red', linestyle='--', label='1% 기준')
    plt.title('총자산 오차율 분포', fontweight='bold')
    plt.xlabel('오차율 (%)')
    plt.ylabel('빈도')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    df['총자산_오차1'].plot(kind='box', vert=False)
    plt.title('총자산 절대 오차 분포', fontweight='bold')
    plt.xlabel('오차 (원)')
    plt.tight_layout()
    plt.show()

### 1.2 총자산 = 부채총계 + 자기자본

In [4]:
# 공식 검증
if all(col in df.columns for col in ['총자산', '부채총계', '자기자본(자본총계)']):
    df['총자산_계산2'] = df['부채총계'] + df['자기자본(자본총계)']
    df['총자산_오차2'] = df['총자산'] - df['총자산_계산2']
    df['총자산_오차율2'] = (df['총자산_오차2'].abs() / df['총자산'] * 100)
    
    print("=== 총자산 = 부채총계 + 자기자본 ===")
    print(f"평균 오차: {df['총자산_오차2'].mean():,.0f}")
    print(f"평균 오차율: {df['총자산_오차율2'].mean():.4f}%")
    print(f"\n오차율 1% 초과: {(df['총자산_오차율2'] > 1).sum():,}건 ({(df['총자산_오차율2'] > 1).sum() / len(df) * 100:.2f}%)")
    
    # 오차가 큰 케이스 확인
    large_error = df[df['총자산_오차율2'] > 5][['총자산', '부채총계', '자기자본(자본총계)', '총자산_오차2', '총자산_오차율2']].head(10)
    if len(large_error) > 0:
        print("\n오차율 5% 초과 샘플:")
        print(large_error)

### 1.3 유동자산 = 당좌자산 + 재고자산

In [5]:
if all(col in df.columns for col in ['유동자산', '당좌자산', '재고자산']):
    df['유동자산_계산'] = df['당좌자산'] + df['재고자산']
    df['유동자산_오차'] = df['유동자산'] - df['유동자산_계산']
    df['유동자산_오차율'] = (df['유동자산_오차'].abs() / df['유동자산'] * 100)
    
    print("=== 유동자산 = 당좌자산 + 재고자산 ===")
    print(f"평균 오차율: {df['유동자산_오차율'].mean():.4f}%")
    print(f"오차율 1% 초과: {(df['유동자산_오차율'] > 1).sum():,}건 ({(df['유동자산_오차율'] > 1).sum() / len(df) * 100:.2f}%)")

=== 유동자산 = 당좌자산 + 재고자산 ===
평균 오차율: 0.0000%
오차율 1% 초과: 0건 (0.00%)


### 1.4 현금성자산 = 현금 + 현금등가물

In [6]:
if all(col in df.columns for col in ['현금성자산', '현금', '현금등가물']):
    df['현금성자산_계산'] = df['현금'] + df['현금등가물']
    df['현금성자산_오차'] = df['현금성자산'] - df['현금성자산_계산']
    
    # 현금성자산이 0인 경우 제외
    valid_mask = df['현금성자산'] > 0
    df.loc[valid_mask, '현금성자산_오차율'] = (df.loc[valid_mask, '현금성자산_오차'].abs() / df.loc[valid_mask, '현금성자산'] * 100)
    
    print("=== 현금성자산 = 현금 + 현금등가물 ===")
    print(f"평균 오차율: {df.loc[valid_mask, '현금성자산_오차율'].mean():.4f}%")
    print(f"오차율 1% 초과: {(df.loc[valid_mask, '현금성자산_오차율'] > 1).sum():,}건")

=== 현금성자산 = 현금 + 현금등가물 ===
평균 오차율: 103.9392%
오차율 1% 초과: 221,574건


### 1.5 부채총계 = 유동부채 + 비유동부채

In [7]:
if all(col in df.columns for col in ['부채총계', '유동부채', '비유동부채']):
    df['부채총계_계산'] = df['유동부채'] + df['비유동부채']
    df['부채총계_오차'] = df['부채총계'] - df['부채총계_계산']
    df['부채총계_오차율'] = (df['부채총계_오차'].abs() / df['부채총계'] * 100)
    
    print("=== 부채총계 = 유동부채 + 비유동부채 ===")
    print(f"평균 오차율: {df['부채총계_오차율'].mean():.4f}%")
    print(f"오차율 1% 초과: {(df['부채총계_오차율'] > 1).sum():,}건 ({(df['부채총계_오차율'] > 1).sum() / len(df) * 100:.2f}%)")

=== 부채총계 = 유동부채 + 비유동부채 ===
평균 오차율: 0.0000%
오차율 1% 초과: 0건 (0.00%)


## 2. 손익계산서 공식 검증

### 2.1 매출총이익 = 매출액 - 매출원가

In [8]:
if all(col in df.columns for col in ['매출총이익', '매출액', '매출원가']):
    df['매출총이익_계산'] = df['매출액'] - df['매출원가']
    df['매출총이익_오차'] = df['매출총이익'] - df['매출총이익_계산']
    
    valid_mask = df['매출총이익'] != 0
    df.loc[valid_mask, '매출총이익_오차율'] = (df.loc[valid_mask, '매출총이익_오차'].abs() / df.loc[valid_mask, '매출총이익'].abs() * 100)
    
    print("=== 매출총이익 = 매출액 - 매출원가 ===")
    print(f"평균 오차율: {df.loc[valid_mask, '매출총이익_오차율'].mean():.4f}%")
    print(f"오차율 1% 초과: {(df.loc[valid_mask, '매출총이익_오차율'] > 1).sum():,}건")

=== 매출총이익 = 매출액 - 매출원가 ===
평균 오차율: 0.0000%
오차율 1% 초과: 0건


### 2.2 영업이익 = 매출총이익 - 판매비와관리비

In [9]:
if all(col in df.columns for col in ['영업손익', '매출총이익', '판매비와관리비']):
    df['영업이익_계산'] = df['매출총이익'] - df['판매비와관리비']
    df['영업이익_오차'] = df['영업손익'] - df['영업이익_계산']
    
    valid_mask = df['영업손익'] != 0
    df.loc[valid_mask, '영업이익_오차율'] = (df.loc[valid_mask, '영업이익_오차'].abs() / df.loc[valid_mask, '영업손익'].abs() * 100)
    
    print("=== 영업이익 = 매출총이익 - 판매비와관리비 ===")
    print(f"평균 오차율: {df.loc[valid_mask, '영업이익_오차율'].mean():.4f}%")
    print(f"오차율 1% 초과: {(df.loc[valid_mask, '영업이익_오차율'] > 1).sum():,}건")

=== 영업이익 = 매출총이익 - 판매비와관리비 ===
평균 오차율: 0.0000%
오차율 1% 초과: 0건


## 3. 재무비율 계산 검증 및 재계산

### 3.1 부채비율 = (부채총계 / 자기자본) × 100

In [10]:
if all(col in df.columns for col in ['재무비율_부채비율', '부채총계', '자기자본(자본총계)']):
    # 자기자본이 0이 아닌 경우만
    valid_mask = (df['자기자본(자본총계)'] != 0) & df['재무비율_부채비율'].notna()
    
    # 재계산
    df.loc[valid_mask, '부채비율_재계산'] = (df.loc[valid_mask, '부채총계'] / df.loc[valid_mask, '자기자본(자본총계)']) * 100
    df['부채비율_차이'] = (df['재무비율_부채비율'] - df['부채비율_재계산']).abs()
    
    print("=== 부채비율 검증 ===")
    print(f"평균 차이: {df.loc[valid_mask, '부채비율_차이'].mean():.2f}%p")
    print(f"차이 5%p 초과: {(df.loc[valid_mask, '부채비율_차이'] > 5).sum():,}건 ({(df.loc[valid_mask, '부채비율_차이'] > 5).sum() / valid_mask.sum() * 100:.2f}%)")
    
    # 제공값 vs 계산값 비교
    plt.figure(figsize=(10, 6))
    sample = df[valid_mask].sample(min(1000, valid_mask.sum()))
    plt.scatter(sample['재무비율_부채비율'], sample['부채비율_재계산'], alpha=0.3, s=1)
    plt.plot([0, 500], [0, 500], 'r--', label='y=x')
    plt.xlim(0, 500)
    plt.ylim(0, 500)
    plt.xlabel('제공된 부채비율 (%)')
    plt.ylabel('재계산 부채비율 (%)')
    plt.title('부채비율: 제공값 vs 계산값', fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 3.2 유동비율 = (유동자산 / 유동부채) × 100

In [11]:
if all(col in df.columns for col in ['재무비율_유동비율', '유동자산', '유동부채']):
    valid_mask = (df['유동부채'] != 0) & df['재무비율_유동비율'].notna()
    
    df.loc[valid_mask, '유동비율_재계산'] = (df.loc[valid_mask, '유동자산'] / df.loc[valid_mask, '유동부채']) * 100
    df['유동비율_차이'] = (df['재무비율_유동비율'] - df['유동비율_재계산']).abs()
    
    print("=== 유동비율 검증 ===")
    print(f"평균 차이: {df.loc[valid_mask, '유동비율_차이'].mean():.2f}%p")
    print(f"차이 5%p 초과: {(df.loc[valid_mask, '유동비율_차이'] > 5).sum():,}건 ({(df.loc[valid_mask, '유동비율_차이'] > 5).sum() / valid_mask.sum() * 100:.2f}%)")

=== 유동비율 검증 ===
평균 차이: 0.00%p
차이 5%p 초과: 0건 (0.00%)


### 3.3 영업이익률 = (영업이익 / 매출액) × 100

In [12]:
if all(col in df.columns for col in ['재무비율_영업이익율', '영업손익', '매출액']):
    valid_mask = (df['매출액'] != 0) & df['재무비율_영업이익율'].notna()
    
    df.loc[valid_mask, '영업이익률_재계산'] = (df.loc[valid_mask, '영업손익'] / df.loc[valid_mask, '매출액']) * 100
    df['영업이익률_차이'] = (df['재무비율_영업이익율'] - df['영업이익률_재계산']).abs()
    
    print("=== 영업이익률 검증 ===")
    print(f"평균 차이: {df.loc[valid_mask, '영업이익률_차이'].mean():.2f}%p")
    print(f"차이 1%p 초과: {(df.loc[valid_mask, '영업이익률_차이'] > 1).sum():,}건 ({(df.loc[valid_mask, '영업이익률_차이'] > 1).sum() / valid_mask.sum() * 100:.2f}%)")

=== 영업이익률 검증 ===
평균 차이: 0.00%p
차이 1%p 초과: 0건 (0.00%)


## 4. 파생 변수 생성 (Feature Engineering)

### 4.1 기본 파생 변수

In [13]:
print("=== 파생 변수 생성 ===")

# 1. 순운전자본 = 유동자산 - 유동부채
if all(col in df.columns for col in ['유동자산', '유동부채']):
    df['순운전자본_재계산'] = df['유동자산'] - df['유동부채']
    print("✓ 순운전자본 생성")

# 2. 순차입금 = 차입금 - 현금성자산
if all(col in df.columns for col in ['차입금', '현금성자산']):
    df['순차입금_재계산'] = df['차입금'] - df['현금성자산']
    print("✓ 순차입금 생성")

# 3. 당좌비율 = (당좌자산 / 유동부채) × 100
if all(col in df.columns for col in ['당좌자산', '유동부채']):
    valid_mask = df['유동부채'] != 0
    df.loc[valid_mask, '당좌비율_재계산'] = (df.loc[valid_mask, '당좌자산'] / df.loc[valid_mask, '유동부채']) * 100
    print("✓ 당좌비율 생성")

# 4. 자기자본비율 = (자기자본 / 총자산) × 100
if all(col in df.columns for col in ['자기자본(자본총계)', '총자산']):
    valid_mask = df['총자산'] != 0
    df.loc[valid_mask, '자기자본비율_재계산'] = (df.loc[valid_mask, '자기자본(자본총계)'] / df.loc[valid_mask, '총자산']) * 100
    print("✓ 자기자본비율 생성")

# 5. 총자산회전율 = 매출액 / 총자산
if all(col in df.columns for col in ['매출액', '총자산']):
    valid_mask = df['총자산'] != 0
    df.loc[valid_mask, '총자산회전율_재계산'] = df.loc[valid_mask, '매출액'] / df.loc[valid_mask, '총자산']
    print("✓ 총자산회전율 생성")

print(f"\n총 생성된 파생 변수: 5개")

=== 파생 변수 생성 ===
✓ 순운전자본 생성
✓ 순차입금 생성
✓ 당좌비율 생성

총 생성된 파생 변수: 5개


### 4.2 추가 재무비율 생성

In [14]:
# ROA (Return on Assets) = (당기순이익 / 총자산) × 100
if all(col in df.columns for col in ['당기순이익', '총자산']):
    valid_mask = df['총자산'] != 0
    df.loc[valid_mask, 'ROA_계산'] = (df.loc[valid_mask, '당기순이익'] / df.loc[valid_mask, '총자산']) * 100
    print("✓ ROA 생성")

# ROE (Return on Equity) = (당기순이익 / 자기자본) × 100
if all(col in df.columns for col in ['당기순이익', '자기자본(자본총계)']):
    valid_mask = df['자기자본(자본총계)'] != 0
    df.loc[valid_mask, 'ROE_계산'] = (df.loc[valid_mask, '당기순이익'] / df.loc[valid_mask, '자기자본(자본총계)']) * 100
    print("✓ ROE 생성")

# 매출총이익률 = (매출총이익 / 매출액) × 100
if all(col in df.columns for col in ['매출총이익', '매출액']):
    valid_mask = df['매출액'] != 0
    df.loc[valid_mask, '매출총이익률_계산'] = (df.loc[valid_mask, '매출총이익'] / df.loc[valid_mask, '매출액']) * 100
    print("✓ 매출총이익률 생성")

# 당기순이익률 = (당기순이익 / 매출액) × 100
if all(col in df.columns for col in ['당기순이익', '매출액']):
    valid_mask = df['매출액'] != 0
    df.loc[valid_mask, '당기순이익률_계산'] = (df.loc[valid_mask, '당기순이익'] / df.loc[valid_mask, '매출액']) * 100
    print("✓ 당기순이익률 생성")

✓ 매출총이익률 생성
✓ 당기순이익률 생성


### 4.3 EBIT / EBITDA 계산 (검증)

In [15]:
# EBIT = 영업이익 + 이자비용 (간소화 공식)
# 또는 EBIT = 법인세차감전순이익 + 금융비용

if all(col in df.columns for col in ['법인세차감전순이익', '금융비용']):
    df['EBIT_계산'] = df['법인세차감전순이익'] + df['금융비용']
    
    if 'EBIT' in df.columns:
        df['EBIT_차이'] = (df['EBIT'] - df['EBIT_계산']).abs()
        print("=== EBIT 검증 ===")
        print(f"평균 차이: {df['EBIT_차이'].mean():,.0f}")
        print(f"\nEBIT 제공값 통계:")
        print(df['EBIT'].describe())
        print(f"\nEBIT 계산값 통계:")
        print(df['EBIT_계산'].describe())

=== EBIT 검증 ===
평균 차이: 4,409,543

EBIT 제공값 통계:
count    6.000000e+05
mean     1.385084e+06
std      2.380708e+08
min     -2.306893e+10
25%     -1.998650e+05
50%      4.226200e+04
75%      4.662015e+05
max      6.465096e+10
Name: EBIT, dtype: float64

EBIT 계산값 통계:
count    6.000000e+05
mean     5.430801e+06
std      3.555855e+08
min     -2.024750e+10
25%     -4.379650e+04
50%      1.343945e+05
75%      9.204290e+05
max      9.427505e+10
Name: EBIT_계산, dtype: float64


### 4.4 성장률 계산

In [16]:
# 매출액 증가율 = ((당기매출 - 전기매출) / 전기매출) × 100
if all(col in df.columns for col in ['매출액', '전기매출액']):
    valid_mask = (df['전기매출액'] != 0) & df['전기매출액'].notna()
    df.loc[valid_mask, '매출액증가율_계산'] = ((df.loc[valid_mask, '매출액'] - df.loc[valid_mask, '전기매출액']) / df.loc[valid_mask, '전기매출액']) * 100
    print("✓ 매출액증가율 생성")

# 영업이익 증가율
if all(col in df.columns for col in ['영업손익', '전기영업이익']):
    valid_mask = (df['전기영업이익'] != 0) & df['전기영업이익'].notna()
    df.loc[valid_mask, '영업이익증가율_계산'] = ((df.loc[valid_mask, '영업손익'] - df.loc[valid_mask, '전기영업이익']) / df.loc[valid_mask, '전기영업이익'].abs()) * 100
    print("✓ 영업이익증가율 생성")

# 총자산 증가율
if all(col in df.columns for col in ['총자산', '자산총계(전기)']):
    valid_mask = (df['자산총계(전기)'] != 0) & df['자산총계(전기)'].notna()
    df.loc[valid_mask, '총자산증가율_계산'] = ((df.loc[valid_mask, '총자산'] - df.loc[valid_mask, '자산총계(전기)']) / df.loc[valid_mask, '자산총계(전기)']) * 100
    print("✓ 총자산증가율 생성")

✓ 매출액증가율 생성
✓ 영업이익증가율 생성


## 5. 생성된 변수 요약

In [17]:
# 생성된 컬럼 목록
generated_cols = [
    col for col in df.columns 
    if any(keyword in col for keyword in ['재계산', '계산', '_차이', '_오차'])
]

print(f"=== 생성된 변수 목록 ({len(generated_cols)}개) ===")
for i, col in enumerate(generated_cols, 1):
    print(f"{i}. {col}")

=== 생성된 변수 목록 (28개) ===
1. 유동자산_계산
2. 유동자산_오차
3. 유동자산_오차율
4. 현금성자산_계산
5. 현금성자산_오차
6. 현금성자산_오차율
7. 부채총계_계산
8. 부채총계_오차
9. 부채총계_오차율
10. 매출총이익_계산
11. 매출총이익_오차
12. 매출총이익_오차율
13. 영업이익_계산
14. 영업이익_오차
15. 영업이익_오차율
16. 유동비율_재계산
17. 유동비율_차이
18. 영업이익률_재계산
19. 영업이익률_차이
20. 순운전자본_재계산
21. 순차입금_재계산
22. 당좌비율_재계산
23. 매출총이익률_계산
24. 당기순이익률_계산
25. EBIT_계산
26. EBIT_차이
27. 매출액증가율_계산
28. 영업이익증가율_계산


## 6. 재계산 값 vs 제공 값 결정

In [18]:
# 주요 재무비율: 제공값과 계산값 중 선택
# 전략: 차이가 크면 재계산값 사용, 작으면 제공값 사용

print("=== 재무비율 사용 전략 ===")

# 부채비율
if '부채비율_차이' in df.columns:
    avg_diff = df['부채비율_차이'].mean()
    if avg_diff > 5:
        print(f"부채비율: 재계산값 사용 (평균 차이 {avg_diff:.2f}%p)")
        df['부채비율_최종'] = df['부채비율_재계산']
    else:
        print(f"부채비율: 제공값 사용 (평균 차이 {avg_diff:.2f}%p)")
        df['부채비율_최종'] = df['재무비율_부채비율']

# 유동비율
if '유동비율_차이' in df.columns:
    avg_diff = df['유동비율_차이'].mean()
    if avg_diff > 5:
        print(f"유동비율: 재계산값 사용 (평균 차이 {avg_diff:.2f}%p)")
        df['유동비율_최종'] = df['유동비율_재계산']
    else:
        print(f"유동비율: 제공값 사용 (평균 차이 {avg_diff:.2f}%p)")
        df['유동비율_최종'] = df['재무비율_유동비율']

# 영업이익률
if '영업이익률_차이' in df.columns:
    avg_diff = df['영업이익률_차이'].mean()
    if avg_diff > 1:
        print(f"영업이익률: 재계산값 사용 (평균 차이 {avg_diff:.2f}%p)")
        df['영업이익률_최종'] = df['영업이익률_재계산']
    else:
        print(f"영업이익률: 제공값 사용 (평균 차이 {avg_diff:.2f}%p)")
        df['영업이익률_최종'] = df['재무비율_영업이익율']

=== 재무비율 사용 전략 ===
유동비율: 제공값 사용 (평균 차이 0.00%p)
영업이익률: 제공값 사용 (평균 차이 0.00%p)


## 7. 분석용 데이터셋 준비

In [19]:
# 분석에 사용할 핵심 컬럼 선택
core_cols = [
    # 기본 정보
    '기준년월', '업종(중분류)', '외감구분', '설립일자', '종업원수',
    
    # 재무상태표
    '총자산', '유동자산', '비유동자산', '부채총계', '자기자본(자본총계)',
    '유동부채', '비유동부채', '현금성자산',
    
    # 손익계산서
    '매출액', '매출원가', '매출총이익', '영업손익', '당기순이익',
    'EBIT', 'EBITDA',
    
    # 재계산된 파생 변수
    '순운전자본_재계산', '순차입금_재계산',
    '부채비율_최종', '유동비율_최종', '영업이익률_최종',
    'ROA_계산', 'ROE_계산',
    '매출액증가율_계산', '총자산증가율_계산',
    
    # 타겟
    '기업신용평가등급(구간화)', '모형개발용Performance(향후1년내부도여부)'
]

# 존재하는 컬럼만 선택
available_cols = [col for col in core_cols if col in df.columns]

df_analysis = df[available_cols].copy()

print(f"=== 분석용 데이터셋 ===")
print(f"Shape: {df_analysis.shape}")
print(f"컬럼 수: {len(available_cols)}")
print(f"\n선택된 컬럼:")
for col in available_cols:
    print(f"  - {col}")

=== 분석용 데이터셋 ===
Shape: (600000, 25)
컬럼 수: 25

선택된 컬럼:
  - 기준년월
  - 업종(중분류)
  - 외감구분
  - 설립일자
  - 종업원수
  - 유동자산
  - 비유동자산
  - 부채총계
  - 유동부채
  - 비유동부채
  - 현금성자산
  - 매출액
  - 매출원가
  - 매출총이익
  - 영업손익
  - 당기순이익
  - EBIT
  - EBITDA
  - 순운전자본_재계산
  - 순차입금_재계산
  - 유동비율_최종
  - 영업이익률_최종
  - 매출액증가율_계산
  - 기업신용평가등급(구간화)
  - 모형개발용Performance(향후1년내부도여부)


## 8. 저장 (선택)

In [20]:
# 분석용 데이터 저장
# df_analysis.to_csv('../data/기업신용평가_분석용.csv', index=False, encoding='utf-8-sig')
# print("✅ 분석용 데이터 저장 완료: data/기업신용평가_분석용.csv")

# 전체 데이터 (파생 변수 포함) 저장
# df.to_csv('../data/기업신용평가_파생변수포함.csv', index=False, encoding='utf-8-sig')
# print("✅ 전체 데이터 저장 완료: data/기업신용평가_파생변수포함.csv")

## 요약

### 주요 발견사항
1. **재무상태표 공식**: 대부분 1% 이내 오차
2. **재무비율 계산**: 제공값과 계산값이 다소 차이 (합성 데이터 특성)
3. **파생 변수**: 추가 재무비율 생성으로 분석 풍부화

### 권장사항
- 재계산한 재무비율 사용 권장
- 파생 변수로 Feature Engineering
- 집단 통계 분석에 적합